In [1]:
from pathlib import Path
import hashlib
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [20]:
!git clone https://github.com/NVlabs/stylegan3.git /content/stylegan3

Cloning into '/content/stylegan3'...
remote: Enumerating objects: 212, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 212 (delta 101), reused 96 (delta 96), pack-reused 46 (from 1)
Receiving objects: 100% (212/212), 4.16 MiB | 18.69 MiB/s, done.
Resolving deltas: 100% (107/107), done.


In [21]:

TRAIN_CSV = Path(
    "/content/drive/MyDrive/bachelor_thesis_data/splits/train_split.csv"
)

#temporary folder
OUTPUT_FOLDER = Path("/content/gan_melanoma_512")

STYLEGAN_REPO = Path("/content/stylegan3")

OUTPUT_ZIP = Path(
    "/content/drive/MyDrive/bachelor_thesis_data/gan/"
    "melanoma_train_512.zip"
)

train_df = pd.read_csv(TRAIN_CSV)

print(f"Total training images: {len(train_df)}")
print(train_df.columns.tolist())

melanoma_df = (
    train_df[train_df["label"] == 1]
    .copy()
    .reset_index(drop=True)
)

print(f"Training melanoma images selected: {len(melanoma_df)}")

assert len(melanoma_df) == 943, (
    f"Expected 943 melanoma images, but found {len(melanoma_df)}."
)

if OUTPUT_FOLDER.exists():
    shutil.rmtree(OUTPUT_FOLDER)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

for _, row in tqdm(
    melanoma_df.iterrows(),
    total=len(melanoma_df),
    desc="Preparing GAN images"
):
    image_id = str(row["isic_id"])
    source_path = Path(row["image_path"])

    if not source_path.exists():
        raise FileNotFoundError(
            f"Image not found: {source_path}"
        )

    destination_path = OUTPUT_FOLDER / f"{image_id}.png"

    with Image.open(source_path) as image:
        image = ImageOps.exif_transpose(image)

        image = image.convert("RGB")

        image = ImageOps.fit(
            image,
            (512, 512),
            method=Image.Resampling.LANCZOS
        )

        image.save(destination_path, format="PNG")


processed_images = list(OUTPUT_FOLDER.glob("*.png"))

print(f"Processed images saved: {len(processed_images)}")

assert len(processed_images) == 943

dataset_tool = STYLEGAN_REPO / "dataset_tool.py"

assert dataset_tool.exists(), (
    f"dataset_tool.py not found at {dataset_tool}. "
    "Clone the StyleGAN2-ADA repository first."
)

OUTPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

command = [
    sys.executable,
    str(dataset_tool),
    f"--source={OUTPUT_FOLDER}",
    f"--dest={OUTPUT_ZIP}"
]

subprocess.run(command, check=True)

print("\nGAN dataset successfully created.")
print(f"Folder: {OUTPUT_FOLDER}")
print(f"StyleGAN ZIP: {OUTPUT_ZIP}")

Total training images: 8204
['isic_id', 'lesion_id', 'diagnosis_3', 'binary_label', 'label', 'image_path']
Training melanoma images selected: 943


Preparing GAN images:   0%|          | 0/943 [00:00<?, ?it/s]

Processed images saved: 943

GAN dataset successfully created.
Folder: /content/gan_melanoma_512
StyleGAN ZIP: /content/drive/MyDrive/bachelor_thesis_data/gan/melanoma_train_512.zip


In [15]:
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git /content/stylegan2-ada-pytorch

Cloning into '/content/stylegan2-ada-pytorch'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 131 (delta 0), reused 0 (delta 0), pack-reused 129 (from 2)
Receiving objects: 100% (131/131), 1.13 MiB | 18.98 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [16]:
from pathlib import Path

dataset_tool = Path(
    "/content/stylegan2-ada-pytorch/dataset_tool.py"
)

print("dataset_tool exists:", dataset_tool.exists())

dataset_tool exists: True


In [17]:
from pathlib import Path
import subprocess
import sys

OUTPUT_FOLDER = Path("/content/gan_melanoma_256")

OUTPUT_ZIP = Path(
    "/content/drive/MyDrive/bachelor_thesis_data/"
    "GAN/melanoma_train_256.zip"
)

OUTPUT_ZIP.parent.mkdir(
    parents=True,
    exist_ok=True
)

dataset_tool = Path(
    "/content/stylegan2-ada-pytorch/dataset_tool.py"
)

command = [
    sys.executable,
    str(dataset_tool),
    f"--source={OUTPUT_FOLDER}",
    f"--dest={OUTPUT_ZIP}"
]

subprocess.run(command, check=True)

print("StyleGAN dataset created:")
print(OUTPUT_ZIP)

StyleGAN dataset created:
/content/drive/MyDrive/bachelor_thesis_data/GAN/melanoma_train_256.zip


In [18]:
print("ZIP exists:", OUTPUT_ZIP.exists())
print(
    "ZIP size:",
    round(OUTPUT_ZIP.stat().st_size / (1024 ** 2), 2),
    "MB"
)

ZIP exists: True
ZIP size: 177.26 MB


In [19]:
import zipfile
from pathlib import Path

OUTPUT_ZIP = Path(
    "/content/drive/MyDrive/bachelor_thesis_data/"
    "GAN/melanoma_train_256.zip"
)

with zipfile.ZipFile(OUTPUT_ZIP, "r") as zip_file:
    files = zip_file.namelist()

    image_files = [
        file for file in files
        if file.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    print("Images inside ZIP:", len(image_files))
    print("Contains dataset.json:", "dataset.json" in files)

Images inside ZIP: 943
Contains dataset.json: True
